# Tool Use Mini Demo - Local Weather Tool

This short notebook gives you a simple introduction to tool use before the main forensic case. In this warm-up, you will see the pattern in a familiar setting: ask a question, call a tool, inspect the result, and answer from evidence instead of guessing.


## What You Will Do

1. Load the lab environment
2. Define one small local weather tool
3. Call the tool manually
4. See how `ToolAgent` invokes the tool
5. Let `ToolAgent` complete the full run


### Step 1: Set Up the Notebook

This cell does the basic setup work for the demo. It:

- reads this lab's `.env` file so the notebook knows which model to use
- connects to the model server
- adds the local `src/` folder so the notebook can import the course tool code

You do not need to memorize every line here. The main idea is that the notebook must load the model settings before it can run any tool demo. Open the notebook from the `lab2_tool_use_pattern` folder so this setup works as expected.


In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and builds the model client.
LAB_NAME = 'lab2_tool_use_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        'Expected .env in this folder. Copy .env.example to .env first.'
    )

load_dotenv(env_path, override=True)

MODEL = os.getenv("MODEL")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f"MODEL or OLLAMA_BASE_URL is missing from {env_path}")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


Repo root: /home/frank/projects/agentic-AI4-forensics
Lab folder: /home/frank/projects/agentic-AI4-forensics/lab2_tool_use_pattern


## Part 0: Mini Tool-Use Warm-Up

Before you work on the vehicle case, start with one tiny familiar example. The goal is to see the tool-use loop in the simplest possible form:

1. ask a question the model should not guess
2. call a tool to get outside information
3. use that tool output to answer

This warm-up uses a small local weather tool. It is only a teaching example, not part of the graded forensic report.


### Step 2: Define the Tool

This cell defines one very small tool called `get_weather`. A tool is just a function the model is allowed to call when it needs outside information.

What to notice:

- the tool has a clear name
- the tool expects specific inputs: `city` and `date`
- the tool returns structured information instead of a vague sentence
- the `@tool` wrapper makes the function available to `ToolAgent`
- after `@tool`, `get_weather` is a `Tool` object, so we use `get_weather.run(...)` to execute it

This is the key idea behind tool use: the model should ask the tool for information instead of guessing.


In [2]:
from agentic_patterns.tool_pattern.tool import tool, validate_arguments
from agentic_patterns.tool_pattern.tool_agent import ToolAgent

weather_lookup = {
    ("Seattle", "2026-01-03"): {
        "condition": "rain showers",
        "high_f": 46,
        "low_f": 39,
        "precipitation_chance": 0.85,
    },
    ("Phoenix", "2026-01-03"): {
        "condition": "sunny",
        "high_f": 68,
        "low_f": 47,
        "precipitation_chance": 0.00,
    },
    ("Boston", "2026-01-03"): {
        "condition": "cloudy",
        "high_f": 34,
        "low_f": 26,
        "precipitation_chance": 0.20,
    },
}

@tool
def get_weather(city: str, date: str):
    """
    Return a small local weather forecast from a mock lookup table.

    Args:
        city (str): City name such as "Seattle".
        date (str): Date in YYYY-MM-DD format.
    """
    normalized_city = city.strip().title()
    normalized_date = date.strip()
    record = weather_lookup.get((normalized_city, normalized_date))
    if record is None:
        return json.dumps(
            {"error": f"No weather record found for {normalized_city} on {normalized_date}"},
            indent=2,
        )
    return json.dumps(
        {"city": normalized_city, "date": normalized_date, **record},
        indent=2,
    )

warmup_question = "Should the detective bring an umbrella to a witness interview in Seattle on 2026-01-03?"

{
    "warmup_question": warmup_question,
    "tool_signature": json.loads(get_weather.fn_signature),
}


{'warmup_question': 'Should the detective bring an umbrella to a witness interview in Seattle on 2026-01-03?',
 'tool_signature': {'name': 'get_weather',
  'description': '\n    Return a small local weather forecast from a mock lookup table.\n\n    Args:\n        city (str): City name such as "Seattle".\n        date (str): Date in YYYY-MM-DD format.\n    ',
  'parameters': {'properties': {'city': {'type': 'str'},
    'date': {'type': 'str'}}}}}

### Step 3: Call the Tool Yourself

This cell shows manual tool use. You run the tool directly with specific arguments and inspect the result.

This is helpful because it separates two jobs:

- the tool gathers the outside information
- you can later turn that information into a plain-language answer

In other words, you are doing the tool-calling step yourself before asking the agent to do it. This example stops at the observation so you can clearly see what evidence the answer should come from.


In [3]:
# Because `@tool` wrapped the function into a Tool object, we call `.run(...)` to execute it.
# Then we turn its JSON text output into a Python dictionary.
manual_weather_observation = json.loads(get_weather.run(city="Seattle", date="2026-01-03"))

# Show the observation returned by the tool.
manual_weather_observation


{'city': 'Seattle',
 'date': '2026-01-03',
 'condition': 'rain showers',
 'high_f': 46,
 'low_f': 39,
 'precipitation_chance': 0.85}

### Step 4: See How `ToolAgent` Invokes the Tool

This cell shows the core function-call step inside `ToolAgent`. Pretend the model has already suggested a tool call. `ToolAgent` then:

- validates the tool-call arguments
- looks up the tool by name in `tools_dict`
- executes the wrapped function with `chosen_tool.run(**validated_tool_call["arguments"])`

This is the moment where the weather function actually gets invoked.


In [ ]:
weather_tool_agent = ToolAgent(tools=[get_weather], client=client, model=MODEL)

# Pretend the model has already returned this tool-call dictionary.
example_tool_call = {
    "name": "get_weather",
    "arguments": {"city": "Seattle", "date": "2026-01-03"},
    "id": 1,
}

# `ToolAgent` validates the arguments before it runs the tool.
validated_tool_call = validate_arguments(example_tool_call, json.loads(get_weather.fn_signature))

# Next it finds the requested tool by name and invokes it.
chosen_tool = weather_tool_agent.tools_dict[validated_tool_call["name"]]
tool_result = chosen_tool.run(**validated_tool_call["arguments"])

{
    "tool_call_from_model": example_tool_call,
    "validated_tool_call": validated_tool_call,
    "selected_tool": chosen_tool.name,
    "tool_result": json.loads(tool_result),
}


### Step 5: Let `ToolAgent` Complete the Full Run

Now that you have seen the exact invocation step, this cell shows the full end-to-end version. The model decides the tool call, `ToolAgent` runs it, and the model writes the final answer from the returned observation.

Compare the three views in this notebook:

- in Step 3, you ran the tool yourself
- in Step 4, you saw the exact line where `ToolAgent` invokes the tool
- in Step 5, the agent handles the whole process for you


In [4]:
weather_tool_agent = ToolAgent(tools=[get_weather], client=client, model=MODEL)

weather_tool_agent_prompt = f"""
Use the available weather tool to answer the user's question.

Question:
{warmup_question}

Return a short answer that cites the tool result and does not guess beyond it.
""".strip()

weather_tool_agent_answer = weather_tool_agent.run(user_msg=weather_tool_agent_prompt)
print(weather_tool_agent_answer)



Using Tool: get_weather

Tool call dict: 
{'name': 'get_weather', 'arguments': {'city': 'Seattle', 'date': '2026-01-03'}, 'id': 1}

Tool result: 
{
  "city": "Seattle",
  "date": "2026-01-03",
  "condition": "rain showers",
  "high_f": 46,
  "low_f": 39,
  "precipitation_chance": 0.85
}
The detective should bring an umbrella. The weather forecast for Seattle on 2026-01-03 indicates an 85% chance of precipitation, suggesting rain showers are likely.


## Next Step

Before you move on, notice that every step in this notebook used the same weather tool and the same evidence. The main difference is how much of the tool-calling work you did yourself versus how much `ToolAgent` handled for you.

Now move to `02_case_overview.md`, then open `03b_lab_notebook.ipynb` to apply the same idea to the vehicle-sale case.
